# LAB 09: Transformer Fine-Tuning & Evaluation for NER
**Học phần:** Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Mô hình:** Pretrained Transformer (BERT-base-NER) fine-tuned on CoNLL-2003


## 1. Kiến Trúc Transformer Token Classification
- Đầu vào: Chuỗi văn bản $X = (x_1, \dots, x_T)$
- Contextual Representation: $H \in \mathbb{R}^{B \times T \times d}$ (với $d=768$)
- Classification Head: $Z = HW + b \in \mathbb{R}^{B \times T \times C}$ (với $C=9$)
- Loss: Cross-Entropy bỏ qua các vị trí nhãn `-100` (padding và subwords thừa).


In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import load_dataset

# Khởi tạo mô hình và tokenizer
model_checkpoint = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)
model.eval()

ds = load_dataset("lhoestq/conll2003")
print("Mô hình và tập dữ liệu đã sẵn sàng!")
print(f"Số lượng nhãn (C): {model.config.num_labels}")
print(f"Nhãn id2label: {model.config.id2label}")


## 2. Experiment 2: Baseline vs Pretrained Transformer
So sánh hiệu năng giữa mô hình Baseline (Linear Classifier với đặc trưng thủ công) và Pretrained Transformer trên cùng tập kiểm thử test set CoNLL-2003.


In [ ]:
results_df = pd.read_csv("results.csv")
print("BẢNG SO SÁNH HIỆU NĂNG TRÊN TẬP TEST (Entity-Level):")
print(results_df.to_string(index=False))


## 3. Experiment 3: Entity-Level Evaluation
- **Token-level accuracy:** Chỉ kiểm tra xem từng token độc lập có đúng nhãn hay không. Metric này không phù hợp cho NER vì token `O` chiếm áp đảo ($>85\%$) và không phản ánh tính toàn vẹn của thực thể.
- **Entity-level evaluation (chuẩn CoNLL / seqeval):** Một thực thể chỉ được tính là đúng (True Positive) khi **cả loại nhãn và toàn bộ ranh giới span** được dự đoán chính xác tuyệt đối.  
  *Ví dụ:* Với nhãn vàng `New York` (`B-LOC I-LOC`), nếu mô hình dự đoán `New York` (`B-LOC O`), token `New` đúng nhưng toàn bộ thực thể `New York` bị coi là sai.


## 4. Experiment 4: Entity Type Analysis
Phân tích chi tiết hiệu năng theo từng loại thực thể: `PER`, `ORG`, `LOC`, `MISC`.


In [ ]:
entity_df = pd.read_csv("entity_metrics.csv")
print("KẾT QUẢ ĐÁNH GIÁ THEO TỪNG LOẠI THỰC THỂ (Entity Metrics):")
print(entity_df.to_string(index=False))


### Giải thích yêu cầu: Vì sao performance giữa các loại entity có thể khác nhau?
1. **Số lượng mẫu huấn luyện (Training Examples):** `PER` (Person) và `LOC` (Location) xuất hiện với tần suất lớn nhất và có tính đại diện cao trong tập train, giúp mô hình học được đặc trưng tốt hơn và đạt F1 cao nhất (>92-96%).
2. **Khuôn mẫu từ vựng (Lexical Patterns):** Tên người thường có cấu trúc 2 từ viết hoa (`First Last`), tên địa danh thường là các danh từ riêng cố định. Ngược lại, `MISC` (Miscellaneous) là nhóm thực thể hỗn hợp (quốc tịch, sự kiện, giải đấu, tác phẩm nghệ thuật), không có cấu trúc thống nhất nên khó học hơn (F1 chỉ ~77%).
3. **Tính nhập nhằng (Ambiguity) & Ngữ cảnh (Context):** `ORG` (Organization) có tính đa nghĩa cao (thường trùng tên với địa danh hoặc cá nhân, ví dụ: *Barcelona* vừa là thành phố vừa là câu lạc bộ), đòi hỏi mô hình phải dựa mạnh vào văn cảnh hành động xung quanh.
4. **Ranh giới thực thể (Entity Boundary):** Các tổ chức (`ORG`) và sự kiện (`MISC`) thường gồm nhiều từ ghép phức tạp chứa giới từ (*"of"*, *"for"*), làm tăng nguy cơ sai ranh giới so với tên người hoặc địa danh ngắn.
5. **Miền văn bản (Domain):** Dữ liệu CoNLL-2003 là tin tức thời sự Reuters 1996, phong cách viết báo chí thể thao chi phối nhiều danh từ riêng viết hoa trong headlines, gây nhiễu cho các lớp ít phổ biến.


## 5. Experiment 5: Entity Length Analysis
Phân chia thực thể theo độ dài (1 token, 2 tokens, 3 tokens, 4+ tokens) và đánh giá sự thay đổi của F1 score.


In [ ]:
from IPython.display import Image, display

# Hiển thị biểu đồ phân tích độ dài thực thể
display(Image(filename="entity_length_analysis.png"))


### Nhận xét thực nghiệm về độ dài thực thể:
- Thực thể có độ dài $1$ và $2$ tokens chiếm đại đa số ($>90\%$) và có F1 score rất cao (~90-95%) do cấu trúc tên riêng ngắn điển hình (*"Clinton"*, *"New York"*).
- Khi độ dài tăng lên $3$ hoặc $4+$ tokens, F1 score suy giảm đáng kể (xuống ~84%) do xác suất duy trì chính xác toàn bộ chuỗi nhãn liên tục `[B-X, I-X, I-X, ...]` giảm theo cấp số nhân (compounding error) và sự xuất hiện của các từ nối chức năng (`of`, `and`, `the`).


## 6. Confusion Matrix
Trực quan hóa ma trận nhầm lẫn (Normalized Confusion Matrix) để kiểm tra các lỗi gán nhãn token phổ biến nhất.


In [ ]:
display(Image(filename="confusion_matrix.png"))
